In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import urllib.parse



load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}"
)

In [4]:
df = pd.read_sql("SELECT * FROM customer_behavior_raw", engine)
df.head()

,user_id,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,...,cart_items_average,checkout_abandonments_per_month,purchase_conversion_rate,app_usage_frequency,notification_response_rate,account_age_months,last_purchase_date,social_sharing_frequency,premium_subscription,return_rate
0,1,56,Female,Germany,Suburban,90860,Self-employed,Associate Degree,Single,0,...,10,2,62,7,74,19,2025-06-22,6,1,50
1,2,69,Male,Japan,Suburban,35423,Unemployed,Bachelor,Single,1,...,5,7,54,5,23,8,2026-07-25,3,0,37
2,3,46,Female,India,Urban,21467,Self-employed,Associate Degree,Married,1,...,3,3,33,7,12,13,2026-02-26,6,0,53
3,4,32,Male,Canada,Urban,41770,Self-employed,Bachelor,Widowed,0,...,5,9,26,4,19,9,2026-10-27,7,0,98
4,5,60,Female,Japan,Urban,183882,Employed,Associate Degree,Widowed,1,...,8,0,18,7,30,3,2026-06-23,3,0,86


In [5]:
print("Shape:", df.shape)
df.info()

Shape: (1000000, 60)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 60 columns):
 #   Column                           Non-Null Count    Dtype 
---  ------                           --------------    ----- 
 0   user_id                          1000000 non-null  int64 
 1   age                              1000000 non-null  int64 
 2   gender                           1000000 non-null  object
 3   country                          1000000 non-null  object
 4   urban_rural                      1000000 non-null  object
 5   income_level                     1000000 non-null  int64 
 6   employment_status                1000000 non-null  object
 7   education_level                  1000000 non-null  object
 8   relationship_status              1000000 non-null  object
 9   has_children                     1000000 non-null  int64 
 10  household_size                   1000000 non-null  int64 
 11  occupation                       1000000 no

In [5]:
df.isnull().sum().sort_values(ascending=False).head(10)

user_id                         0
age                             0
impulse_buying_score            0
environmental_consciousness     0
health_conscious_shopping       0
travel_frequency                0
hobby_count                     0
social_media_influence_score    0
reading_habits                  0
exercise_frequency              0
dtype: int64

In [6]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns


Index(['user_id', 'age', 'gender', 'country', 'urban_rural', 'income_level',
       'employment_status', 'education_level', 'relationship_status',
       'has_children', 'household_size', 'occupation', 'ethnicity',
       'language_preference', 'device_type', 'weekly_purchases',
       'monthly_spend', 'cart_abandonment_rate', 'review_writing_frequency',
       'average_order_value', 'preferred_payment_method',
       'coupon_usage_frequency', 'loyalty_program_member', 'referral_count',
       'product_category_preference', 'shopping_time_of_day',
       'weekend_shopper', 'impulse_purchases_per_month', 'browse_to_buy_ratio',
       'return_frequency', 'budgeting_style', 'brand_loyalty_score',
       'impulse_buying_score', 'environmental_consciousness',
       'health_conscious_shopping', 'travel_frequency', 'hobby_count',
       'social_media_influence_score', 'reading_habits', 'exercise_frequency',
       'stress_from_financial_decisions', 'overall_stress_level',
       'sleep_quali

In [7]:
df.dtypes.head(15)


user_id                object
age                    object
gender                 object
country                object
urban_rural            object
income_level           object
employment_status      object
education_level        object
relationship_status    object
has_children           object
household_size         object
occupation             object
ethnicity              object
language_preference    object
device_type            object
dtype: object

In [8]:

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

df.dtypes.value_counts()


C:\Users\Nandini\AppData\Local\Temp\ipykernel_25792\898250148.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


int64     45
object    15
Name: count, dtype: int64

In [9]:
df.select_dtypes(include="object").columns


Index(['gender', 'country', 'urban_rural', 'employment_status',
       'education_level', 'relationship_status', 'occupation', 'ethnicity',
       'language_preference', 'device_type', 'preferred_payment_method',
       'product_category_preference', 'shopping_time_of_day',
       'budgeting_style', 'last_purchase_date'],
      dtype='object')

In [10]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.lower()


In [11]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 35, 50, 70],
    labels=["18-25", "26-35", "36-50", "51+"]
)


In [12]:
df["income_band"] = pd.qcut(
    df["income_level"],
    q=3,
    labels=["low", "medium", "high"]
)


In [13]:
df["engagement_score"] = (
    df["social_media_influence_score"] +
    df["impulse_buying_score"] +
    df["exercise_frequency"]
)


In [6]:
df.to_sql(
    "customer_behavior_cleaned",
    engine,
    if_exists="replace",
    index=False
)

print("customer_behavior_cleaned table created ✅")


customer_behavior_cleaned table created ✅


In [16]:
pd.read_sql("SELECT COUNT(*) FROM customer_behavior_cleaned", engine)


,count
0,1000000


In [17]:
df.groupby("age_group")["engagement_score"].mean().sort_values(ascending=False)


C:\Users\Nandini\AppData\Local\Temp\ipykernel_25792\927967821.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("age_group")["engagement_score"].mean().sort_values(ascending=False)


age_group
26-35    13.518909
36-50    13.503868
51+      13.498179
18-25    13.489171
Name: engagement_score, dtype: float64

In [18]:
df.groupby("income_band")["impulse_buying_score"].mean()


C:\Users\Nandini\AppData\Local\Temp\ipykernel_25792\2475501770.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("income_band")["impulse_buying_score"].mean()


income_band
low       5.007071
medium    4.995005
high      4.993193
Name: impulse_buying_score, dtype: float64

In [19]:
df["recommended_category"] = df.apply(
    lambda row: "electronics"
    if row["impulse_buying_score"] > 7
    else "home",
    axis=1
)
